# Credit Risk Classification 
Includes EDA, Models, GridSearch, Comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.cluster import KMeans


In [ ]:
url = 'https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv'
df = pd.read_csv(url)
df.head()

In [ ]:
print(df.info())
print(df.isnull().sum())

## EDA

In [ ]:
sns.countplot(x='risk', data=df)
plt.title('Risk Distribution')
plt.show()

In [ ]:
sns.histplot(df['amount'], bins=30, kde=True)
plt.title('Loan Amount Distribution')
plt.show()

In [ ]:
sns.boxplot(x='risk', y='age', data=df)
plt.title('Age vs Risk')
plt.show()

In [ ]:
sns.boxplot(x='risk', y='duration', data=df)
plt.title('Duration vs Risk')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## Preprocessing

In [ ]:
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

In [ ]:
X = df.drop('risk', axis=1)
y = df['risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Models

In [ ]:
# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
print('Logistic Accuracy:', lr_acc)

# KNN
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)
print('KNN Accuracy:', knn_acc)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
print('Decision Tree Accuracy:', dt_acc)

# Random Forest
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print('Random Forest Accuracy:', rf_acc)

## GridSearch (Improvement)

In [ ]:
params = {'n_neighbors': [3,5,7,9]}
grid = GridSearchCV(KNeighborsClassifier(), params, cv=5)
grid.fit(X_train, y_train)

best_knn = grid.best_estimator_
best_pred = best_knn.predict(X_test)
best_acc = accuracy_score(y_test, best_pred)

print('Best Params:', grid.best_params_)
print('Improved KNN Accuracy:', best_acc)

## Confusion Matrices

In [ ]:
sns.heatmap(confusion_matrix(y_test, lr_pred), annot=True)
plt.title('Logistic Confusion Matrix')
plt.show()

sns.heatmap(confusion_matrix(y_test, best_pred), annot=True)
plt.title('Improved KNN Confusion Matrix')
plt.show()

## Model Comparison

In [ ]:
models = ['Logistic', 'KNN', 'Decision Tree', 'Random Forest', 'Improved KNN']
scores = [lr_acc, knn_acc, dt_acc, rf_acc, best_acc]

sns.barplot(x=models, y=scores)
plt.title('Model Comparison')
plt.ylabel('Accuracy')
plt.show()

## Clustering

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X)

plt.scatter(X[:,0], X[:,1], c=clusters)
plt.title('K-Means Clustering')
plt.show()

## Save Model

In [ ]:
import pickle
pickle.dump(rf, open('model.pkl','wb'))
print('Model saved successfully')